# Emergency profiles

Emergency profiles group past emergencies by the kind of damage they caused, so a new emergency can be described by the group it resembles. The groups come from K-means clustering on the logarithm of the nine impact variables.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"

## Train the profiles

`train_profiles` tries several numbers of clusters and keeps the best by silhouette score. Only emergencies with a recorded impact are clustered.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from ml.data import load_frame
from ml.features import IMPACT_VARIABLES, recorded_impact
from ml.profiles import train_profiles

frame = load_frame(DATA / "ungrd_emergencies_v2.csv", DATA / "divipola_dane.csv")
result = train_profiles(frame)
metrics = result.metrics
print(f"Chosen number of profiles (k): {metrics['k']}")
print(f"Silhouette: {metrics['silhouette']:.3f}")

## Silhouette by number of clusters

The silhouette score measures how well separated the clusters are. Higher is better.

In [ ]:
by_k = pd.Series(metrics["silhouette_by_k"])
ax = by_k.plot(marker="o", figsize=(6, 3.5))
ax.set_xlabel("Number of clusters (k)")
ax.set_ylabel("Silhouette")
plt.show()

## Stability

Stability is the average adjusted Rand index between the chosen clustering and clusterings started from different random seeds. A value close to 1 means the groups do not depend on the seed.

In [ ]:
print(f"Stability (adjusted Rand index): {metrics['stability_ari']:.3f}")

## Profile sizes

The number of emergencies in each profile, plus the emergencies that report no impact.

In [ ]:
sizes = pd.Series(metrics["profile_sizes"])
print(sizes.to_string())
print(f"No recorded impact: {metrics['no_impact_rows']}")

## Average impact per profile

The mean of each impact variable per profile, in original units (not logarithms).

In [ ]:
model = result.model
frame = frame.assign(profile=model.assign(frame))
clustered = frame[frame["has_impact"] & recorded_impact(frame)]
clustered.groupby("profile")[list(IMPACT_VARIABLES)].mean().round(2)

## Top event types per profile

The three most frequent event types in each profile.

In [ ]:
for profile, group in clustered.groupby("profile"):
    print(f"{profile}: {group['event_type'].value_counts().head(3).to_dict()}")